# 03 - EELS-Auswertung Praktikumsdatensatz

Gleiches Vorgehen wie Notebook 01, aber an einem anderen Datensatz: eine
FIB-Lamelle mit deutlich mehr Elementen (Si, O, Ga, W, Pt, S). Das ist der
Datensatz aus dem TEM-Praktikum.

**Arbeite Notebook 01 zuerst durch** - dort sind die einzelnen Schritte erklaert,
hier stehen nur noch die Besonderheiten.

In [ ]:
# Interaktive Plots (zoomen, Spektrum je Bildpunkt anklicken).
# Falls die Plots weiss bleiben oder gar nichts erscheint:
# diese Zeile durch  %matplotlib inline  ersetzen und den Kernel neu starten.
%matplotlib widget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

import hyperspy.api as hs
import exspy  # muss importiert sein, sonst kennt HyperSpy die EELS-/EDX-Signaltypen nicht

# Findet die Messdaten unabhaengig vom Betriebssystem (siehe workshop_data.py)
from workshop_data import load, load_standards

print("HyperSpy", hs.__version__, "| exspy", exspy.__version__)

## 1. Daten laden

In [ ]:
signal = load("praktikum_eels_highloss", signal_type="EELS")
ll = load("praktikum_eels_lowloss", signal_type="EELS")

signal.plot()

In [ ]:
load("praktikum_adf").plot()

## 2. Zero-Loss-Peak ausrichten

In [ ]:
ll.align_zero_loss_peak(also_align=[signal], signal_range=(-10.0, 10.0))

## 3. Modell aufbauen

Sechs Elemente statt drei - entsprechend mehr Kanten, entsprechend laengerer Fit.
Deshalb hier `rebin` mit **8x8** statt 2x2. Das ist eine bewusste Abwaegung:
Ortsaufloesung gegen Rechenzeit. Wenn du Zeit hast, probiere ruhig 4x4.

Achte auf ueberlappende Kanten (Ga-L bei ~1115 eV, W-N, Pt-N...). Wenn der Fit
unruhig wird, ist meist eine Kante schuld, die im gemessenen Energiebereich gar
nicht vollstaendig enthalten ist.

In [ ]:
signal.add_elements(["Si", "O", "Ga", "W", "Pt", "S"])

signal_binned = signal.rebin(scale=[8, 8, 1])
signal_binned = signal_binned.remove_background(signal_range=(70.0, 96.0))

m = signal_binned.create_model(auto_background=False)
m.components

In [ ]:
# --- Variante A: interaktiv ---
m.gui()

In [ ]:
# --- Variante B: per Code ---
for komponente in m:
    print(f"{komponente.name}   aktiv={komponente.active}")

In [ ]:
m.plot()

In [ ]:
m.multifit(kind="smart")

In [ ]:
m.plot_results()

## 4. Feinstruktur mit Si-Referenzspektren

**Hinweis:** Die Referenzspektren stammen aus dem Nanopore-Datensatz, nicht aus
dieser Messung. Das war schon im urspruenglichen Notebook so (dort allerdings
ueber einen Pfad, der nur auf einem Rechner existierte). Solange die
Aufnahmebedingungen vergleichbar sind, ist das vertretbar - man sollte es aber
wissen, wenn man die Ergebnisse interpretiert.

In [ ]:
signal_binned = signal.rebin(scale=[2, 2, 1])
signal_binned = signal_binned.remove_background(signal_range=(70.0, 96.0))
signal_binned = signal_binned.isig[92.0:170.0]

s_smooth = signal_binned.deepcopy()
s_smooth.data = gaussian_filter1d(s_smooth.data, sigma=2, axis=-1)
s_smooth.plot()

In [ ]:
standards = load_standards("si_standards", sigma=2)

for name, s in standards.items():
    s.data = s.data / s.data.max()
    print(name)

In [ ]:
m = s_smooth.create_model(auto_background=False)

for name, s in standards.items():
    muster = hs.model.components1D.ScalableFixedPattern(s)
    muster.name = name
    muster.xscale.free = False
    muster.shift.free = False
    muster.yscale.bmin = 0
    muster.yscale.bmax = 1e7
    m.append(muster)

m.components

In [ ]:
# --- Variante A: interaktiv ---
m.gui()

In [ ]:
# --- Variante B: per Code ---
for komponente in m:
    print(f"{komponente.name:20s} yscale={komponente.yscale.value}")

In [ ]:
m.plot()

In [ ]:
m.multifit(bounded=True)

In [ ]:
m.plot_results()

## Aufgaben

1. Setze `rebin` auf `[4, 4, 1]`. Wie viel laenger dauert der Fit, und siehst du
   in den Elementkarten wirklich mehr?
2. Nimm einzelne Elemente aus `add_elements` heraus. Bei welchen wird der Fit
   sichtbar schlechter, bei welchen aendert sich fast nichts - und was sagt dir das?
3. Vergleiche die Ga-Verteilung mit dem ADF-Bild. Wo sitzt das Gallium, und
   woher kommt es bei einer FIB-praeparierten Lamelle?